# Framework + R0 Baseline

## Step 1: How the existing repair loop works (chinatravel/agent/nesy_verifier/`llm_modulo.py`)

Main Method: `LLMModuloAgent.solve(problem, prob_idx, oracle_verifier)`.
There is no separate repair function: repair is just the model re-prompted with the previous
plan + the verifier's error messages.

**Flow:**

1. **Collect + shortlist** candidate transport / hotels / attractions / restaurants from the
   environment. These shortlisted strings are reused in every prompt.
2. **Generate** the initial plan with `INITIAL_PROMPT` (step 0).
3. **Verify** with the oracle verifier →
   `collect_commonsense_constraints_error` + `collect_personal_error`.
   Returns `error_info`: a list of error strings (empty list = valid plan).
4. **Repair loop** `for step_i in range(1, max_steps + 1)`:
   - stop early if `error_info` is empty,
   - re-prompt with `BACK_PROMPT` (previous plan + joined errors) → **whole new plan**,
   - verify again, repeat.
5. Returns `(True, evaluated_plan)`; success is judged from `evaluated_plan["error_info"]`.

**Key points:**

- `max_steps` = repair budget (`--refine_steps`, default 10).
- Stops on success **or** when the budget runs out.
- `BACK_PROMPT` regenerates the **entire** plan each attempt — no slot is protected. This is
  exactly the R0 baseline.
- Feedback = raw error strings, no structured constraint selection.

**Which function does what:**

| Step | Where |
|---|---|
| Initial generation | `INITIAL_PROMPT` + `self.backbone_llm(...)` |
| Verification | `collect_commonsense_constraints_error` + `collect_personal_error` → `error_info` |
| Feedback | join `error_info` into `error_info_nl` |
| Regeneration | `BACK_PROMPT` + `self.backbone_llm(...)` inside the loop |
| Retry limit | `for step_i in range(1, max_steps + 1)` with `break` when `error_info` is empty |

## Step 2: Setting Up and Testing R0

### 2.1 Environment Setup

Set up ChinaTravel using Python 3.12.8 and installed the dependencies from `requirements.txt`.

We are using:
- **Dataset:** Original ChinaTravel dataset from Hugging Face
- **Sandbox:** ChinaTravel-Sandbox (Chinese)
- **LLM:** GPT-4.1 through OpenRouter
- **Baseline:** LLM-Modulo (R0)

Encountered a few issues during setup:
1. **Dependency issues:** PyArrow was incompatible with NumPy 1.26.4, so I downgraded PyArrow to 19.0.1. Python was also missing the `_lzma` module, which was fixed by reinstalling Python with XZ support.
2. **Dataset format:** `hard_logic_py` was stored as a string instead of a list. Modified `load_datasets.py` to convert it using `ast.literal_eval()`.
3. **Missing sandbox:** Downloaded the original ChinaTravel-Sandbox from Hugging Face and extracted it into `chinatravel/environment/database/`.
4. **Missing import:** Added `import sys` to `llm_modulo.py`.
5. **OpenRouter configuration:** Updated `load_model.py` so that the full model identifier `openai/gpt-4.1` is passed to OpenRouter.

### 2.2 Testing R0

Ran R0 using the original ChinaTravel dataset with `--refine_steps 2`.

**Test 1 — Easy split**

Case: `e20241028160248698752`

- One-day trip from Shanghai to Hangzhou, budget 1,500 RMB.
- The initial itinerary passed the reported commonsense checks.
- No repair was needed (`ITER 0 ERROR: []`).
- Confirmed that initial generation and verification work.

**Test 2 — Medium split**

Case: `e20241028160845920703`

- Two-day trip from Shenzhen to Guangzhou, budget 1,000 RMB, hotpot preference and single-bed room.
- Initial itinerary failed a timing constraint: dinner on Day 2 ended too early.
- R0 used the verifier feedback to regenerate the itinerary.
- After one repair, the verifier returned `ITER 1 ERROR: []`.

This confirms that the R0 repair loop works for the observed timing violation.

### 2.3 Logging

Checked the output files generated by ChinaTravel.

For each iteration, it saves:
- `step_0_prompt.txt` — Initial prompt
- `step_0_plan.json` — Initial itinerary and verification results
- `step_0_error.err` — Initial errors
- `step_1_...` — Corresponding files after repair

The saved plans also contain token usage and execution time.

However, the existing logs do not directly track which constraints were fixed or newly violated after each repair. We will need to add this for our experiments.

### 2.4 Findings and Issues to Follow Up

- R0 successfully generates, verifies and repairs itineraries.
- R0 regenerates the whole itinerary without protecting previously satisfied constraints.
- The current verifier returns error messages rather than structured constraint results.
- The logs show `no hard_logic_nl information`, so we still need to check whether personal constraints are being verified properly.
- The setup changes will need to be documented so teammates can reproduce the environment.

## Step 3: Shared Framework Implementation

I separated the repair process from the original ChinaTravel agent so that R0–R4 can use the same planner and verifier without rewriting everything.

### 3.1 Files Added

| File | Purpose |
|---|---|
| `shared_verifier.py` | Checks commonsense and personal constraints |
| `shared_repair_loop.py` | Handles repair attempts, verification, acceptance/rejection and logging |
| `repair_strategies/base.py` | Defines the common interface for R0–R4 |
| `repair_strategies/r0.py` | Implements R0 whole-plan regeneration |

These files are inside `chinatravel/agent/nesy_verifier/`.

I also updated `run_exp.py`, `load_model.py` and `llm_modulo.py` to support selecting a repair strategy.

### 3.2 How the New Framework Works

1. ChinaTravel generates an initial itinerary.
2. The shared verifier checks commonsense and personal constraints.
3. The selected repair strategy proposes a revised itinerary.
4. The shared verifier checks the proposed repair.
5. The strategy decides whether to accept or reject it.
6. Repeat until the plan passes verification or the repair limit is reached.

The original ChinaTravel repair loop is still available when no repair strategy is selected.

### 3.3 Improvements Added

- **Shared verification:** Fixed the issue where personal constraints were skipped because `hard_logic_nl` was missing. The new verifier uses `hard_logic_py` directly.
- **Raw itinerary preservation:** Keeps the original itinerary separate from the translated transport version.
- **Acceptance/rejection:** Supports rejecting a proposed repair and keeping the previous accepted itinerary.
- **Constraint-transition logging:** Tracks which personal constraints were fixed, newly violated, remained failed or remained satisfied.
- **Error handling:** Personal constraint evaluation errors are recorded separately as `None` instead of automatically counting them as violations.
- **Separate outputs:** R0–R4 have different result and log directories.

### 3.4 Testing

- Tested the original R0 on ChinaTravel easy and medium cases.
- Tested the shared verifier on a saved itinerary.
- Tested the standalone R0 strategy using a mock LLM.
- Ran the new R0 end-to-end on a medium case.
- All 7 shared-framework unit tests passed.

In the new R0 end-to-end test, reported errors decreased from 38 → 3 → 2 after two repair attempts. However, the final itinerary still had violations.

**Current status:** The basic shared framework and R0 are working. Further verifier reliability testing is needed before final experiments.

---

## Step 4: How to start on your side

### 4.1 Main Files to Look At

Start with these files:

1. `repair_strategies/base.py` — Understand the repair interface.
2. `repair_strategies/r0.py` — Use this as an example implementation.
3. `shared_repair_loop.py` — See how repairs are verified and accepted.
4. `shared_verifier.py` — See what verification results are available.

There is no need to rewrite the original ChinaTravel planner.

### 4.2 Implementing Your Repair Strategy

Create a new file inside `repair_strategies/`, for example `r1.py`, `r2.py`, `r3.py` or `r4.py`.

Each strategy should inherit from `BaseRepairStrategy` and implement:

```python
def repair(self, problem, raw_plan, verification_results, context, llm):
    ...
    return RepairResult(
        repaired_plan=new_plan,
        metadata={"strategy": "R1"}
    )
```

**Inputs:**

- `problem`: Original travel request.
- `raw_plan`: Current itinerary before transport translation.
- `verification_results`: Constraint feedback from the shared verifier.
- `context`: Available travel information.
- `llm`: GPT-4.1 adapter.

**Output:** A `RepairResult` containing the proposed itinerary and metadata.

Strategies can also override `accept()` if they need to reject repairs.

### 4.3 Responsibilities

| Strategy | Main task |
|---|---|
| R1 | Repair only the affected itinerary slots |
| R2 | Protect previously satisfied constraints and reject harmful repairs |
| R3 | Start with targeted repair and expand the scope when needed |
| R4 | Use the constraint-interdependency matrix to guide repair |

**Important:** Keep the itinerary in ChinaTravel's expected format. Local transport belongs inside each activity's `transports` field, not as a separate activity.

### 4.4 Running Experiments

The intended command is:

```bash
python run_exp.py \
  --agent LLM-modulo \
  --repair_strategy R0 \
  --llm openai/gpt-4.1 \
  --splits medium \
  --refine_steps 2 \
  --oracle_translation \
  --lang zh
```

Replace `R0` with your strategy after it has been implemented and registered in `llm_modulo.py`.

Currently, only the new R0 strategy is connected. R1–R4 still need to be registered and tested.

### 4.5 Things to Take Note Of

- Don't modify the original ChinaTravel repair loop unless necessary.
- Use the same shared verifier and repair budget for fair comparisons.
- Rejected repairs should still count toward the repair budget.
- Check both constraint satisfaction and newly introduced violations.
- The current commonsense verifier can still produce execution errors, and transport translation can fail silently. These need to be addressed before final evaluation.
- Test your strategy with a mock LLM before making paid API calls.